# Autoresearch Experiment Analysis

Analysis of autonomous EMR model tuning results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, val_bce_loss, memory_gb, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["val_bce_loss"] = pd.to_numeric(df["val_bce_loss"], errors="coerce")
df["memory_gb"]    = pd.to_numeric(df["memory_gb"],    errors="coerce")
df["status"]       = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep    = counts.get("KEEP",    0)
n_discard = counts.get("DISCARD", 0)
n_crash   = counts.get("CRASH",   0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    bce  = row["val_bce_loss"]
    desc = row["description"]
    print(f"  #{i:3d}  val_bce={bce:.6f}  mem={row['memory_gb']:.1f}GB  {desc}")

## Val BCE Loss Over Time

Track how the best (kept) `val_bce_loss` evolves as experiments progress.
The running minimum shows the frontier — the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes
valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)

baseline_bce = valid.loc[0, "val_bce_loss"]
best         = valid[valid["status"] == "KEEP"]["val_bce_loss"].min()

# Only plot experiments near or below baseline
below = valid[valid["val_bce_loss"] <= baseline_bce + 0.005]

# Discarded as faint background dots
disc = below[below["status"] == "DISCARD"]
ax.scatter(disc.index, disc["val_bce_loss"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Kept experiments as prominent green dots
kept_v = below[below["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["val_bce_loss"],
           c="#2ecc71", s=50, zorder=4, label="Kept",
           edgecolors="black", linewidths=0.5)

# Running minimum step line
kept_mask   = valid["status"] == "KEEP"
kept_idx    = valid.index[kept_mask]
kept_bce    = valid.loc[kept_mask, "val_bce_loss"]
running_min = kept_bce.cummin()
ax.step(kept_idx, running_min, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its description
for idx, bce in zip(kept_idx, kept_bce):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, bce),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept  = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation BCE Loss (lower is better)", fontsize=12)
ax.set_title(f"Autoresearch EMR Progress: {n_total} Experiments, {n_kept} Kept Improvements",
             fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

margin = max((baseline_bce - best) * 0.15, 0.001)
ax.set_ylim(best - margin, baseline_bce + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
kept         = df[df["status"] == "KEEP"].copy()
baseline_bce = df.iloc[0]["val_bce_loss"]
best_bce     = kept["val_bce_loss"].min()
best_row     = kept.loc[kept["val_bce_loss"].idxmin()]

print(f"Baseline val_bce_loss:  {baseline_bce:.6f}")
print(f"Best val_bce_loss:      {best_bce:.6f}")
print(f"Total improvement:      {baseline_bce - best_bce:.6f}  "
      f"({(baseline_bce - best_bce) / baseline_bce * 100:.2f}%)")
print(f"Best experiment:        {best_row['description']}")
print()

print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: val_bce={row['val_bce_loss']:.6f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is vs the previous kept experiment
kept = df[df["status"] == "KEEP"].copy()
kept["prev_bce"] = kept["val_bce_loss"].shift(1)
kept["delta"]    = kept["prev_bce"] - kept["val_bce_loss"]

hits = kept.iloc[1:].copy().sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'BCE Loss':>10}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_bce_loss']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")